In [10]:
from tucker_tensor import TuckerDecomposition, _role_index, _voc_index, np_sim
import numpy as np
import os
from similarity import load_eval_sentences_cached, load_og_sentences
from utils import DATA_DIR
from typing import Tuple
# now, our code does this:
def check_vocab(tensor, triple: Tuple[str, str, str]) -> bool:
    """Checks if the given (verb, subject, object) triple is in the vocabulary."""
    v_in = triple[0] in tensor.vocab["v2i"]
    s_in = triple[1] in tensor.vocab["s2i"]
    o_in = triple[2] in tensor.vocab["o2i"]
    return (v_in, s_in, o_in)

def evaluate_sample(tensor, sentences):
    total_score = 0
    total_prob_score = 0
    n_samples = len(sentences)
    non_eval = 0
    for tup in sentences:
        (v, s, o) = check_vocab(tensor, tup)
        if not (v and s and o):
            test = [(t, el) for t, el in zip(tup, (s,v,o))]

            # print(test)
            non_eval += 1
            continue
        score = 0
        prob_score = 0
        # print()
        for i, element in enumerate(tup):
            role = ["verb", "subject", "object"][i]
            r2i = {"verb": "v2i", "subject": "s2i", "object": "o2i"}[role]
            G_excluded = tensor.excluded_role_vector(tup, role=role)

            # update to avoid division by 0
            F = tensor.factors[i].cpu().numpy()  # (N,R)

            # --- defensive norm computation ---
            F_norm = np.linalg.norm(F, axis=1)
            G_norm = np.linalg.norm(G_excluded)

            eps = 1e-12  # safeguard lower bound
            F_norm = np.maximum(F_norm, eps)
            G_norm = max(G_norm, eps)

            # --- safe cosine similarities ---
            similarities = (F @ G_excluded) / (F_norm * G_norm)
            top_k = 5
            top_idx = np.argsort(similarities)[-top_k:][::-1]

            # build inverse vocab once per role
            i2w = {i: w for w, i in tensor.vocab[r2i].items()}

            top_items = [(i2w[i], similarities[i]) for i in top_idx]

            print(f"[{role}, {element}] top-{top_k} similarities:")
            for w, sim in top_items:
                print(f"  {w:20s} {sim:.6f}")
            # print(similarities)
            idx = tensor.vocab[r2i][element]
            sim_i = similarities[idx]
            # print(element, sim_i)

            # --- rank = number of items >= the true similarity ---
            rank = np.sum(similarities >= sim_i)

            # safeguard rank for extreme cases
            if rank == 0:
                rank = 1

            score += 1.0 / rank
            prob_score += sim_i
        total_score += score/len(tup)
        total_prob_score += prob_score/3
    average_score = total_score / n_samples
    average_prob_score = total_prob_score / n_samples
    print(f"Could not evaluate {non_eval} out of {n_samples} ({non_eval/n_samples*100}%)")
    print(f"Overall score = {average_score}")
    print(f"True score = {total_score/(n_samples-non_eval)}")
    print(f"Overall prob score = {average_prob_score}")
    print(f"True prob score = {total_prob_score/(n_samples-non_eval)}")
    return average_score

In [11]:
tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_large",
                                            method="sc",
                                            dims=3000,
                                            rank=150,
                                            iterations=1000,
                                            map_location="cpu",
                                            name = "klNoB"
                              )

In [12]:
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_dutch_10000000.csv")
random_state = 1
sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset="fineweb_large",
                                             seed=random_state,
                                             n_samples=10_000,
                                             )

In [13]:
sample_sentences = sentence_sample[:20]
res = evaluate_sample(tensor, sentence_sample)

[verb, benadrukken] top-5 similarities:
  verbreken            0.968350
  hervaten             0.954293
  ingeblazen           0.927747
  ronden               0.830016
  beginnen             0.827211
[subject, show] top-5 similarities:
  resultaat            0.343582
  onderzoek            0.218008
  studie               0.200792
  woordvoerder         0.199156
  None                 0.197189
[object, band] top-5 similarities:
  None                 0.999946
  bronverwijzing       0.006820
  spoiler              0.005795
  tegendeel            0.004301
  tendens              0.003797
[verb, vergeten] top-5 similarities:
  interviewen          0.822693
  gijzelen             0.693818
  arresteren           0.640027
  vermoorden           0.624979
  terugbellen          0.623344
[subject, ik] top-5 similarities:
  ik                   0.604867
  je                   0.487456
  mens                 0.159383
  jij                  0.150064
  u                    0.125666
[object, jongen] t